# Assignment 2: Support Ticket Classifier using Pre-trained Language Models

## Objective
Build an end-to-end transformer-based text classification pipeline using Hugging Face to automatically classify customer support tickets into categories like Billing, Technical, Delivery, etc.

## Tools Used
- Python
- Pandas
- Hugging Face Transformers
- Datasets
- PyTorch
- Scikit-learn

## Dataset
Customer Support Tickets Dataset from Hugging Face: `Tobi-Bueck/customer-support-tickets`

## Step 1: Install Required Libraries

In [ ]:
# Install required packages
!pip install transformers datasets torch scikit-learn pandas numpy accelerate -q

## Step 2: Import Libraries

In [ ]:
import pandas as pd
import numpy as np
from datasets import load_dataset
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer,
    DataCollatorWithPadding,
    pipeline
)
from sklearn.metrics import accuracy_score, precision_recall_fscore_support, classification_report, confusion_matrix
from sklearn.preprocessing import LabelEncoder
import torch
import warnings
warnings.filterwarnings('ignore')

# Check if GPU is available
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

## Step 3: Load the Dataset

In [ ]:
# Load the Customer Support Tickets dataset from Hugging Face
dataset = load_dataset("Tobi-Bueck/customer-support-tickets")
print(dataset)

In [ ]:
# Display the structure of the dataset
print("\nDataset Features:")
print(dataset['train'].features)
print(f"\nNumber of training samples: {len(dataset['train'])}")

In [ ]:
# View sample data
print("\nSample entries from the dataset:")
for i in range(5):
    print(f"\n--- Sample {i+1} ---")
    print(dataset['train'][i])

## Step 4: Explore the Dataset

In [ ]:
# Convert to pandas DataFrame for easier exploration
df = pd.DataFrame(dataset['train'])
print("Dataset Shape:", df.shape)
print("\nColumn Names:")
print(df.columns.tolist())
df.head(10)

In [ ]:
# Check for missing values
print("Missing Values:")
print(df.isnull().sum())

In [ ]:
# Identify the text column and label column
# The column names may vary based on the dataset structure
print("\nDataset Info:")
df.info()

In [ ]:
# Analyze the label distribution
# First, let's identify the label column (could be 'category', 'label', 'ticket_type', etc.)
print("\nColumn value counts for potential label columns:")
for col in df.columns:
    if df[col].dtype == 'object' and df[col].nunique() < 20:
        print(f"\n{col}:")
        print(df[col].value_counts())

## Step 5: Data Preprocessing

In [ ]:
# Based on the dataset exploration, define the text and label columns
# Common column names for this dataset are:
# - text column: 'text', 'ticket', 'description', 'Ticket Description'
# - label column: 'category', 'label', 'ticket_type', 'Ticket Type'

# You may need to adjust these based on actual column names
text_column = None
label_column = None

# Auto-detect columns
for col in df.columns:
    col_lower = col.lower()
    if 'text' in col_lower or 'description' in col_lower or 'ticket' in col_lower:
        if text_column is None and df[col].dtype == 'object' and df[col].str.len().mean() > 20:
            text_column = col
    if 'type' in col_lower or 'category' in col_lower or 'label' in col_lower:
        if label_column is None:
            label_column = col

print(f"Text column: {text_column}")
print(f"Label column: {label_column}")

In [ ]:
# If auto-detection doesn't work, manually set the columns
# Uncomment and modify the lines below if needed:
# text_column = 'your_text_column_name'
# label_column = 'your_label_column_name'

# Display unique labels
if label_column:
    print(f"\nUnique labels in '{label_column}':")
    unique_labels = df[label_column].unique()
    print(unique_labels)
    print(f"\nNumber of unique labels: {len(unique_labels)}")

In [ ]:
# Encode labels to integers
label_encoder = LabelEncoder()
df['label_encoded'] = label_encoder.fit_transform(df[label_column])

# Create label mappings
label2id = {label: idx for idx, label in enumerate(label_encoder.classes_)}
id2label = {idx: label for idx, label in enumerate(label_encoder.classes_)}

print("Label to ID mapping:")
print(label2id)
print("\nID to Label mapping:")
print(id2label)

In [ ]:
# Visualize label distribution
import matplotlib.pyplot as plt

plt.figure(figsize=(12, 6))
df[label_column].value_counts().plot(kind='bar', color='steelblue', edgecolor='black')
plt.title('Distribution of Support Ticket Categories', fontsize=14)
plt.xlabel('Category', fontsize=12)
plt.ylabel('Count', fontsize=12)
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.show()

## Step 6: Prepare Dataset for Training

In [ ]:
from datasets import Dataset, DatasetDict
from sklearn.model_selection import train_test_split

# Prepare the data with text and labels
df_clean = df[[text_column, 'label_encoded']].rename(columns={
    text_column: 'text',
    'label_encoded': 'label'
})

# Remove any rows with missing values
df_clean = df_clean.dropna()

# Split into train and test sets
train_df, test_df = train_test_split(
    df_clean, 
    test_size=0.2, 
    random_state=42, 
    stratify=df_clean['label']
)

# Further split train into train and validation
train_df, val_df = train_test_split(
    train_df, 
    test_size=0.1, 
    random_state=42, 
    stratify=train_df['label']
)

print(f"Training samples: {len(train_df)}")
print(f"Validation samples: {len(val_df)}")
print(f"Test samples: {len(test_df)}")

In [ ]:
# Convert pandas DataFrames to Hugging Face Datasets
train_dataset = Dataset.from_pandas(train_df, preserve_index=False)
val_dataset = Dataset.from_pandas(val_df, preserve_index=False)
test_dataset = Dataset.from_pandas(test_df, preserve_index=False)

# Create a DatasetDict
dataset_dict = DatasetDict({
    'train': train_dataset,
    'validation': val_dataset,
    'test': test_dataset
})

print(dataset_dict)

## Step 7: Load Pre-trained Model and Tokenizer

In [ ]:
# Choose a pre-trained model
# Options: 'bert-base-uncased', 'distilbert-base-uncased', 'roberta-base'
# Using DistilBERT for faster training while maintaining good performance
model_name = 'distilbert-base-uncased'

# Load tokenizer
tokenizer = AutoTokenizer.from_pretrained(model_name)
print(f"Loaded tokenizer: {model_name}")

In [ ]:
# Tokenize the dataset
def tokenize_function(examples):
    return tokenizer(
        examples['text'],
        padding='max_length',
        truncation=True,
        max_length=256  # Adjust based on your text length
    )

# Apply tokenization to all splits
tokenized_datasets = dataset_dict.map(tokenize_function, batched=True)
print("\nTokenized datasets:")
print(tokenized_datasets)

In [ ]:
# Inspect a tokenized sample
sample = tokenized_datasets['train'][0]
print("Sample tokenized entry:")
print(f"Text: {sample['text'][:100]}...")
print(f"Input IDs length: {len(sample['input_ids'])}")
print(f"Label: {sample['label']} ({id2label[sample['label']]})")

In [ ]:
# Load the pre-trained model for sequence classification
num_labels = len(label2id)

model = AutoModelForSequenceClassification.from_pretrained(
    model_name,
    num_labels=num_labels,
    id2label=id2label,
    label2id=label2id
)

print(f"\nModel loaded with {num_labels} labels")
print(f"Model parameters: {model.num_parameters():,}")

## Step 8: Define Training Arguments and Metrics

In [ ]:
# Define compute metrics function
def compute_metrics(eval_pred):
    predictions, labels = eval_pred
    predictions = np.argmax(predictions, axis=1)
    
    accuracy = accuracy_score(labels, predictions)
    precision, recall, f1, _ = precision_recall_fscore_support(
        labels, predictions, average='weighted'
    )
    
    return {
        'accuracy': accuracy,
        'precision': precision,
        'recall': recall,
        'f1': f1
    }

In [ ]:
# Define training arguments
training_args = TrainingArguments(
    output_dir='./results',
    num_train_epochs=3,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    warmup_steps=500,
    weight_decay=0.01,
    logging_dir='./logs',
    logging_steps=100,
    eval_strategy='epoch',
    save_strategy='epoch',
    load_best_model_at_end=True,
    metric_for_best_model='f1',
    greater_is_better=True,
    report_to='none',  # Disable wandb
    push_to_hub=False
)

print("Training arguments configured")

In [ ]:
# Create data collator for dynamic padding
data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

## Step 9: Train the Model

In [ ]:
# Initialize the Trainer
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_datasets['train'],
    eval_dataset=tokenized_datasets['validation'],
    tokenizer=tokenizer,
    data_collator=data_collator,
    compute_metrics=compute_metrics
)

print("Trainer initialized")

In [ ]:
# Train the model
print("Starting training...")
train_result = trainer.train()
print("\nTraining completed!")

In [ ]:
# Display training metrics
print("\nTraining Results:")
print(f"Training Loss: {train_result.training_loss:.4f}")
print(f"Training Runtime: {train_result.metrics['train_runtime']:.2f} seconds")
print(f"Training Samples per Second: {train_result.metrics['train_samples_per_second']:.2f}")

## Step 10: Evaluate the Model

In [ ]:
# Evaluate on validation set
print("Evaluating on validation set...")
val_results = trainer.evaluate(tokenized_datasets['validation'])
print("\nValidation Results:")
for key, value in val_results.items():
    print(f"{key}: {value:.4f}")

In [ ]:
# Evaluate on test set
print("Evaluating on test set...")
test_results = trainer.evaluate(tokenized_datasets['test'])
print("\nTest Results:")
for key, value in test_results.items():
    print(f"{key}: {value:.4f}")

In [ ]:
# Get predictions on test set for detailed analysis
predictions = trainer.predict(tokenized_datasets['test'])
pred_labels = np.argmax(predictions.predictions, axis=1)
true_labels = predictions.label_ids

# Classification Report
print("\n" + "="*60)
print("DETAILED CLASSIFICATION REPORT")
print("="*60)
print(classification_report(
    true_labels, 
    pred_labels, 
    target_names=list(id2label.values())
))

In [ ]:
# Confusion Matrix Visualization
import seaborn as sns

cm = confusion_matrix(true_labels, pred_labels)
plt.figure(figsize=(12, 10))
sns.heatmap(
    cm, 
    annot=True, 
    fmt='d', 
    cmap='Blues',
    xticklabels=list(id2label.values()),
    yticklabels=list(id2label.values())
)
plt.title('Confusion Matrix - Support Ticket Classification', fontsize=14)
plt.xlabel('Predicted Label', fontsize=12)
plt.ylabel('True Label', fontsize=12)
plt.xticks(rotation=45, ha='right')
plt.yticks(rotation=0)
plt.tight_layout()
plt.show()

## Step 11: Save the Model

In [ ]:
# Save the fine-tuned model and tokenizer
model_save_path = './support_ticket_classifier'

trainer.save_model(model_save_path)
tokenizer.save_pretrained(model_save_path)

print(f"Model saved to {model_save_path}")

In [ ]:
# Save label mappings for future inference
import json

label_mappings = {
    'label2id': label2id,
    'id2label': id2label
}

with open(f'{model_save_path}/label_mappings.json', 'w') as f:
    json.dump(label_mappings, f, indent=2)

print("Label mappings saved")

## Step 12: Inference - Classify New Support Tickets

In [ ]:
# Create a classification pipeline for easy inference
classifier = pipeline(
    'text-classification',
    model=model_save_path,
    tokenizer=model_save_path,
    device=0 if torch.cuda.is_available() else -1
)

print("Classification pipeline created")

In [ ]:
# Test with sample support tickets
sample_tickets = [
    "Payment failed twice when trying to checkout",
    "Cannot login to my account, password reset not working",
    "Package was damaged when it arrived",
    "Refund still pending after 10 days",
    "Delivery delayed by more than a week",
    "Product is defective and not working as expected",
    "I need to change my billing address",
    "App keeps crashing when I open it"
]

print("="*60)
print("SUPPORT TICKET CLASSIFICATION RESULTS")
print("="*60)

for ticket in sample_tickets:
    result = classifier(ticket)[0]
    print(f"\nTicket: '{ticket}'")
    print(f"Predicted Category: {result['label']}")
    print(f"Confidence: {result['score']:.2%}")

In [ ]:
# Function for interactive ticket classification
def classify_ticket(ticket_text):
    """
    Classify a support ticket and return the predicted category.
    
    Args:
        ticket_text (str): The support ticket text to classify
        
    Returns:
        dict: Dictionary containing label and confidence score
    """
    result = classifier(ticket_text)[0]
    return {
        'ticket': ticket_text,
        'category': result['label'],
        'confidence': f"{result['score']:.2%}"
    }

# Example usage
test_ticket = "My order hasn't arrived and tracking shows it's stuck"
result = classify_ticket(test_ticket)
print("\nClassification Result:")
for key, value in result.items():
    print(f"{key}: {value}")

## Step 13: Model Performance Summary

In [ ]:
# Summary of model performance
print("="*60)
print("MODEL PERFORMANCE SUMMARY")
print("="*60)
print(f"\nModel: {model_name}")
print(f"Number of Labels: {num_labels}")
print(f"Training Samples: {len(train_df)}")
print(f"Validation Samples: {len(val_df)}")
print(f"Test Samples: {len(test_df)}")
print(f"\nTest Set Performance:")
print(f"  - Accuracy: {test_results['eval_accuracy']:.4f}")
print(f"  - Precision: {test_results['eval_precision']:.4f}")
print(f"  - Recall: {test_results['eval_recall']:.4f}")
print(f"  - F1 Score: {test_results['eval_f1']:.4f}")

## Conclusion

In this assignment, we successfully built an end-to-end **Support Ticket Classifier** using:

1. **Hugging Face Datasets** - To load the customer support tickets dataset
2. **Pre-trained DistilBERT** - A lightweight transformer model for efficient training
3. **Hugging Face Transformers** - For fine-tuning and inference
4. **Scikit-learn** - For evaluation metrics and analysis

### Key Takeaways:
- Pre-trained language models can be easily fine-tuned for domain-specific classification tasks
- The Hugging Face ecosystem provides a seamless pipeline from data loading to deployment
- Transfer learning with transformers achieves strong results even with limited training data
- The classification pipeline enables easy integration into production systems

### Next Steps:
- Experiment with different pre-trained models (BERT, RoBERTa, etc.)
- Implement data augmentation for better generalization
- Deploy the model as a REST API for real-time classification
- Fine-tune hyperparameters for improved performance